In [11]:
import json
import math
import os
import functions_for_experiments
from openai import OpenAI
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from qdrant_client.models import SparseTextEmbedding
from thefuzz import fuzz
from fastembed import SparseTextEmbedding

In [12]:
with open("final_golden_dataset.json", "r", encoding="utf-8") as f:
    golden_data = json.load(f)

In [13]:
n = len(golden_data)

## Baseline

In [4]:
MODEL_UKR = SentenceTransformer('lang-uk/ukr-paraphrase-multilingual-mpnet-base')
COLLECTION_UKR = "ucu_documents_ukr"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12039.39it/s]
XLMRobertaModel LOAD REPORT from: lang-uk/ukr-paraphrase-multilingual-mpnet-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
total_hit_ukr, total_mrr_ukr, total_recall_ukr, total_ndcg_ukr, \
total_hit_reranked_ukr, total_mrr_reranked_ukr, total_recall_reranked_ukr, total_ndcg_reranked_ukr = functions_for_experiments.get_metrics(MODEL_UKR, COLLECTION_UKR)

In [6]:
print("Results for Ukrainian model:")
print(f"Hit Rate @ 5: {round(total_hit_ukr/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_ukr/n, 2)}")
print(f"Recall @ 5: {round(total_recall_ukr/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_ukr/n, 2)}")

Results for Ukrainian model:
Hit Rate @ 5: 0.4
MRR @ 5: 0.32
Recall @ 5: 0.38
NDCG @ 5: 0.32


In [7]:
print("Results for Ukrainian model:")
print(f"Hit Rate @ 5: {round(total_hit_reranked_ukr/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_ukr/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_ukr/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_ukr/n, 2)}")

Results for Ukrainian model:
Hit Rate @ 5: 0.6
MRR @ 5: 0.56
Recall @ 5: 0.55
NDCG @ 5: 0.53


## HyDE

In [8]:
total_hit_ukr_hyde, total_mrr_ukr_hyde, total_recall_ukr_hyde, total_ndcg_ukr_hyde, \
total_hit_reranked_ukr_hyde, total_mrr_reranked_ukr_hyde, total_recall_reranked_ukr_hyde, total_ndcg_reranked_ukr_hyde = functions_for_experiments.get_metrics_hyde(MODEL_UKR, COLLECTION_UKR)

In [9]:
print("Results for Ukrainian model (HyDE):")
print(f"Hit Rate @ 5: {round(total_hit_ukr_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_ukr_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_ukr_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_ukr_hyde/n, 2)}")

Results for Ukrainian model (HyDE):
Hit Rate @ 5: 0.45
MRR @ 5: 0.24
Recall @ 5: 0.4
NDCG @ 5: 0.26


In [10]:
print("Results for Ukrainian model (HyDE):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_ukr_hyde/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_ukr_hyde/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_ukr_hyde/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_ukr_hyde/n, 2)}")

Results for Ukrainian model (HyDE):
Hit Rate @ 5: 0.55
MRR @ 5: 0.46
Recall @ 5: 0.47
NDCG @ 5: 0.44


## Query Transform

In [ ]:
total_hit_ukr_transform, total_mrr_ukr_transform, total_recall_ukr_transform, total_ndcg_ukr_transform, \
total_hit_reranked_ukr_transform, total_mrr_reranked_ukr_transform, total_recall_reranked_ukr_transform, total_ndcg_reranked_ukr_transform = functions_for_experiments.get_metrics_query_transform(MODEL_UKR, COLLECTION_UKR)

In [ ]:
print("Results for Ukrainian model (Query Transform):")
print(f"Hit Rate @ 5: {round(total_hit_ukr_transform/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_ukr_transform/n, 2)}")
print(f"Recall @ 5: {round(total_recall_ukr_transform/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_ukr_transform/n, 2)}")

Results for Ukrainian model (Query Transform):
Hit Rate @ 5: 0.3
MRR @ 5: 0.16
Recall @ 5: 0.28
NDCG @ 5: 0.18


In [ ]:
print("Results for Ukrainian model (Query Transform):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_ukr_transform/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_ukr_transform/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_ukr_transform/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_ukr_transform/n, 2)}")

Results for Ukrainian model (Query Transform):
Hit Rate @ 5: 0.45
MRR @ 5: 0.41
Recall @ 5: 0.37
NDCG @ 5: 0.37


## Hybrid

In [14]:
MODEL_SPARSE = SparseTextEmbedding(model_name="Qdrant/bm25")

In [15]:
total_hit_ukr_sparse, total_mrr_ukr_sparse, total_recall_ukr_sparse, total_ndcg_ukr_sparse, \
total_hit_reranked_ukr_sparse, total_mrr_reranked_ukr_sparse, total_recall_reranked_ukr_sparse, total_ndcg_reranked_ukr_sparse = functions_for_experiments.get_metrics_sparse(MODEL_UKR, COLLECTION_UKR, MODEL_SPARSE)

In [16]:
print("Results for Ukrainian model (Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_ukr_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_ukr_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_ukr_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_ukr_sparse/n, 2)}")

Results for Ukrainian model (Hybrid):
Hit Rate @ 5: 0.55
MRR @ 5: 0.43
Recall @ 5: 0.49
NDCG @ 5: 0.42


In [17]:
print("Results for Ukrainian model (Hybrid):")
print(f"Hit Rate @ 5: {round(total_hit_reranked_ukr_sparse/n, 2)}")
print(f"MRR @ 5: {round(total_mrr_reranked_ukr_sparse/n, 2)}")
print(f"Recall @ 5: {round(total_recall_reranked_ukr_sparse/n, 2)}")
print(f"NDCG @ 5: {round(total_ndcg_reranked_ukr_sparse/n, 2)}")

Results for Ukrainian model (Hybrid):
Hit Rate @ 5: 0.65
MRR @ 5: 0.6
Recall @ 5: 0.59
NDCG @ 5: 0.57
